In [1]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

Saving task4_join_export.csv to task4_join_export.csv


In [2]:
import io

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("File:", filename)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

File: task4_join_export.csv
Rows: 609282
Columns: 7

Column names:
['10248', 'VINET', '2016-07-04', '11', '14', '12', '0.0']

First 5 rows:


,10248,VINET,2016-07-04,11,14,12,0.0
0,10248,VINET,2016-07-04,42,9.8,10,0.0
1,10248,VINET,2016-07-04,72,34.8,5,0.0
2,10249,TOMSP,2016-07-05,14,18.6,9,0.0
3,10249,TOMSP,2016-07-05,51,42.4,40,0.0
4,10250,HANAR,2016-07-08,41,7.7,10,0.0



Data types:
10248           int64
VINET          object
2016-07-04     object
11              int64
14            float64
12              int64
0.0           float64
dtype: object


In [10]:
# Task 7a:
df = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    header=None,
    names=[
        "OrderID",
        "CustomerID",
        "OrderDate",
        "ProductID",
        "UnitPrice",
        "Quantity",
        "Discount"
    ]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())
print("\nData types:")
print(df.dtypes)

Rows: 609283
Columns: 7


,OrderID,CustomerID,OrderDate,ProductID,UnitPrice,Quantity,Discount
0,10248,VINET,2016-07-04,11,14.0,12,0.0
1,10248,VINET,2016-07-04,42,9.8,10,0.0
2,10248,VINET,2016-07-04,72,34.8,5,0.0
3,10249,TOMSP,2016-07-05,14,18.6,9,0.0
4,10249,TOMSP,2016-07-05,51,42.4,40,0.0



Data types:
OrderID         int64
CustomerID     object
OrderDate      object
ProductID       int64
UnitPrice     float64
Quantity        int64
Discount      float64
dtype: object


In [11]:
# Task 7b:
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage.round(2)
})

print("Missing-value report:")
display(missing_report)

Missing-value report:


,Missing Count,Missing Percentage
OrderID,0,0.0
CustomerID,0,0.0
OrderDate,0,0.0
ProductID,0,0.0
UnitPrice,0,0.0
Quantity,0,0.0
Discount,0,0.0


In [12]:
# Task 7c:

# Create a copy for the cleaned dataset
df_clean = df.copy()

# Identify numeric and categorical columns
numeric_columns = df_clean.select_dtypes(include=np.number).columns
categorical_columns = df_clean.select_dtypes(include=["object"]).columns

# Impute numeric columns using the median
for column in numeric_columns:
    if df_clean[column].isnull().any():
        df_clean[column] = df_clean[column].fillna(
            df_clean[column].median()
        )

# Impute categorical columns using the mode
for column in categorical_columns:
    if df_clean[column].isnull().any():
        df_clean[column] = df_clean[column].fillna(
            df_clean[column].mode()[0]
        )

print("Task 7(c) — Imputation completed.")
print("\nNumeric columns imputed using: Median")
print("Categorical columns imputed using: Mode")

Task 7(c) — Imputation completed.

Numeric columns imputed using: Median
Categorical columns imputed using: Mode


In [13]:
# Task 7d:

remaining_missing = df_clean.isnull().sum()

print("Task 7(d) — Missing values after imputation:")
display(remaining_missing.to_frame(name="Remaining Missing Values"))

print("\nTotal remaining missing values:", remaining_missing.sum())

Task 7(d) — Missing values after imputation:


,Remaining Missing Values
OrderID,0
CustomerID,0
OrderDate,0
ProductID,0
UnitPrice,0
Quantity,0
Discount,0



Total remaining missing values: 0


In [14]:
# Task 7e:

# Count duplicate rows before removal
duplicates_before = df_clean.duplicated().sum()

print("Task 7(e) — Duplicate analysis")
print("Duplicate rows before removal:", duplicates_before)

# Remove duplicate rows
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Count duplicate rows after removal
duplicates_after = df_clean.duplicated().sum()

print("Duplicate rows after removal:", duplicates_after)
print("Rows after duplicate removal:", len(df_clean))

Task 7(e) — Duplicate analysis
Duplicate rows before removal: 0
Duplicate rows after removal: 0
Rows after duplicate removal: 609283


In [15]:
# Task 8a:

# Exclude identifiers such as OrderID and ProductID.
# Use the numeric measures for outlier analysis.

continuous_columns = [
    "UnitPrice",
    "Quantity",
    "Discount"
]

print("Task 8(a) — Continuous numeric columns selected:")
print(continuous_columns)

print("\nData types:")
print(df_clean[continuous_columns].dtypes)

print("\nSummary statistics:")
display(df_clean[continuous_columns].describe())

Task 8(a) — Continuous numeric columns selected:
['UnitPrice', 'Quantity', 'Discount']

Data types:
UnitPrice    float64
Quantity       int64
Discount     float64
dtype: object

Summary statistics:


,UnitPrice,Quantity,Discount
count,609283.000000,609283.000000,609283.000000
mean,28.850379,25.503095,0.000199
std,33.565470,14.453939,0.005978
min,2.000000,1.000000,0.000000
25%,13.250000,13.000000,0.000000
50%,19.500000,25.000000,0.000000
75%,33.250000,38.000000,0.000000
max,263.500000,130.000000,0.250000


In [16]:
# Task 8b:

iqr_results = []

for column in continuous_columns:
    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    ).sum()

    iqr_results.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "IQR Outlier Count": outlier_count
    })

iqr_report = pd.DataFrame(iqr_results)

print("Task 8(b) — IQR outlier analysis:")
display(iqr_report)

Task 8(b) — IQR outlier analysis:


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,IQR Outlier Count
0,UnitPrice,13.25,33.25,20.0,-16.75,63.25,31622
1,Quantity,13.00,38.00,25.0,-24.50,75.50,49
2,Discount,0.00,0.00,0.0,0.00,0.00,838


In [17]:
# Task 8c:

zscore_results = []

for column in continuous_columns:
    mean_value = df_clean[column].mean()
    std_value = df_clean[column].std()

    # Calculate Z-scores
    z_scores = (df_clean[column] - mean_value) / std_value

    # Count observations where |Z| > 3
    outlier_count = (z_scores.abs() > 3).sum()

    zscore_results.append({
        "Column": column,
        "Mean": mean_value,
        "Standard Deviation": std_value,
        "Z-score Threshold": 3,
        "Z-score Outlier Count": outlier_count
    })

zscore_report = pd.DataFrame(zscore_results)

print("Task 8(c) — Z-score outlier analysis:")
display(zscore_report)

Task 8(c) — Z-score outlier analysis:


,Column,Mean,Standard Deviation,Z-score Threshold,Z-score Outlier Count
0,UnitPrice,28.850379,33.565470,3,7905
1,Quantity,25.503095,14.453939,3,77
2,Discount,0.000199,0.005978,3,837


In [18]:
# Task 8d:

outlier_comparison = iqr_report[
    ["Column", "IQR Outlier Count"]
].merge(
    zscore_report[
        ["Column", "Z-score Outlier Count"]
    ],
    on="Column"
)

print("Task 8(d) — IQR vs Z-score outlier comparison:")
display(outlier_comparison)

print("\nOutlier detection is reported only; no outliers were removed.")

Task 8(d) — IQR vs Z-score outlier comparison:


,Column,IQR Outlier Count,Z-score Outlier Count
0,UnitPrice,31622,7905
1,Quantity,49,77
2,Discount,838,837



Outlier detection is reported only; no outliers were removed.
